# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

 **My Rule:**<br> If CTR is low, engagement is low, low scroll event rate, position is greater than or equal to 10 and page is not updated for a quite long time than refresh page.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("""
Reason codes:
- low_ctr_and_stale — CTR is low, position is 10+, and content hasn't been updated in a long time
- low_engagement_and_low_scroll — engagement rate and scroll rate are both low, position is 10+
- all_signals_weak — CTR, engagement, and scroll are all low, position is 10+, and content is stale (the full match on your rule)
- stable — none of the above conditions are met, no refresh needed
""")


Reason codes:
- low_ctr_and_stale — CTR is low, position is 10+, and content hasn't been updated in a long time
- low_engagement_and_low_scroll — engagement rate and scroll rate are both low, position is 10+
- all_signals_weak — CTR, engagement, and scroll are all low, position is 10+, and content is stale (the full match on your rule)
- stable — none of the above conditions are met, no refresh needed



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

<h2>2.1: Import Libraries</h2>

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
import duckdb

<h2>2.2: Get data from hugging face Flyrank repo</h2>

In [4]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


<h2>2.3: Taken sample 60day window data from fact_daily</h2>

In [5]:
clients_last_3m = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position,
    ga4_pageviews, ga4_sessions, ga4_engaged_sessions, scroll_events
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-05-30'
      AND client_has_gsc == 'true'
      AND client_has_ga4 == 'true'
      AND gsc_data_available == 'true'
      AND ga4_data_available == 'true'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<h2></h2>

<h2>Checking length</h2>

In [6]:
print(len(clients_last_3m))

1111616


<h2>Check columns</h2>

In [7]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'],
      dtype='object')

<h2>Check NULL impressions if any</h2>

In [8]:
clients_last_3m["gsc_impressions"].isna().sum()

np.int64(0)

<h2>2.4: Load dim_content table data in pandas dataframe</h2>

In [9]:
dimf_content = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_updated_date
    FROM {TABLES['dim_content']}
""").df()

<h2>2.5: Calculate CTR, Engagement rate, Scroll rate, Days since update</h2>

In [10]:
clients_last_3m["ctr"] = clients_last_3m["gsc_clicks"] / clients_last_3m["gsc_impressions"]
clients_last_3m["engagement_rate"] = clients_last_3m["ga4_engaged_sessions"] / clients_last_3m["ga4_sessions"]
clients_last_3m["scroll_rate"] = clients_last_3m["scroll_events"] / clients_last_3m["ga4_pageviews"].replace(0, pd.NA)
clients_last_3m = clients_last_3m.merge(
    dimf_content,
    on=("client_hash_id", "content_hash_id"),
    how="left"
)
reference_date = clients_last_3m["report_date"].max()
clients_last_3m["days_since_update"] = (reference_date - clients_last_3m["content_updated_date"]).dt.days

<h2>Checking columns</h2>

In [11]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events', 'ctr', 'engagement_rate',
       'scroll_rate', 'content_updated_date', 'days_since_update'],
      dtype='object')

<h2>2.6: Drop unnecessary columns</h2>

In [12]:
clients_last_3m = clients_last_3m.drop(columns=['content_updated_date', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'])

<h2>Checking columns</h2>

In [13]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update'],
      dtype='object')

<h2>2.7:
2 Signals: <br>  
1. CTR-vs-position (FlyRank signal)<br>
2. Engagement rate vs Scroll rate </h2>

In [14]:
# SIGNAL 1:
clients_last_3m["position_bucket"] = pd.cut(
    clients_last_3m["gsc_avg_position"],
    bins=[0, 3, 10, 20, 100, 100000],
    labels=["1-3", "4-10", "11-20", "21-100", "100+"]
)

signal1 = clients_last_3m.groupby("position_bucket").agg(
    avg_ctr=("ctr", "mean"),
    n=("content_hash_id", "count")
)
print("=== Signal 1: CTR-vs-position ===")
print(signal1)
print("Verdict: MIXED if avg_ctr decreases as position bucket gets worse but we can see if position crosses 100 then there is increase in average ctr")

# SIGNAL 2:
clients_last_3m["engagement_bucket"] = pd.cut(
    clients_last_3m["engagement_rate"],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=["low", "medium", "high", "very_high"]
)

signal2 = clients_last_3m.groupby("engagement_bucket").agg(
    avg_scroll_rate=("scroll_rate", "mean"),
    n=("content_hash_id", "count")
)
print("=== Signal 2: Engagement rate vs Scroll rate ===")
print(signal2)
print("Verdict: CONFIRMED if engagement is high the average scroll rate will also be high")

/tmp/ipykernel_788/1225796085.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1 = clients_last_3m.groupby("position_bucket").agg(
/tmp/ipykernel_788/1225796085.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2 = clients_last_3m.groupby("engagement_bucket").agg(


=== Signal 1: CTR-vs-position ===
                  avg_ctr       n
position_bucket                  
1-3              0.028330   66002
4-10             0.013217  530906
11-20            0.011634  233780
21-100           0.007033  274790
100+             0.038492     248
Verdict: MIXED if avg_ctr decreases as position bucket gets worse but we can see if position crosses 100 then there is increase in average ctr
=== Signal 2: Engagement rate vs Scroll rate ===
                  avg_scroll_rate      n
engagement_bucket                       
low                       0.15123  25246
medium                   0.368508  23512
high                     0.539958    782
very_high                0.804206  27282
Verdict: CONFIRMED if engagement is high the average scroll rate will also be high


<h2>2.8: Check details of columns = ctr, engagement_rate, scroll_rate, days_since_update to find threshold values</h2>

In [15]:
print(clients_last_3m[clients_last_3m["ctr"] < 0.000005])

        report_date           client_hash_id           content_hash_id  \
0        2026-04-01  client_9958f0a7ae1df715  content_810cf06597918291   
2        2026-04-01  client_9958f0a7ae1df715  content_c4002ce386c98905   
5        2026-04-01  client_9958f0a7ae1df715  content_dc2c2198d9631650   
7        2026-04-01  client_9958f0a7ae1df715  content_28d211a926b33519   
8        2026-04-01  client_9958f0a7ae1df715  content_01abeb8b40591eec   
...             ...                      ...                       ...   
1111606  2026-05-30  client_1a8bf67cad4ee525  content_3811343b165eb63a   
1111609  2026-05-30  client_1a8bf67cad4ee525  content_fadf7ae978fd082e   
1111612  2026-05-30  client_1a8bf67cad4ee525  content_5eb9c0b1de0202d2   
1111613  2026-05-30  client_1a8bf67cad4ee525  content_349c92d5cd468777   
1111614  2026-05-30  client_1a8bf67cad4ee525  content_978d1979c92d5bdb   

         gsc_impressions  gsc_avg_position  ctr  engagement_rate scroll_rate  \
0                      1       

In [16]:
clients_last_3m["engagement_rate"].value_counts()

,count
engagement_rate,
0.000000,1017510
1.000000,27265
0.500000,14167
0.333333,8316
0.250000,5559
...,...
0.041237,1
0.074468,1
0.065421,1


In [17]:
clients_last_3m["scroll_rate"].value_counts()

,count
scroll_rate,
0.0,927122
1.0,59581
0.5,42139
0.25,14407
0.333333,14260
...,...
0.348837,1
0.082873,1
0.019504,1


In [18]:
clients_last_3m["days_since_update"].value_counts()

,count
days_since_update,
10,269663
94,169000
-12,78984
12,77713
-23,48275
...,...
218,2
363,2
67,2


<h2>2.9: Threshold values</h2>

In [19]:
ctr_threshold = 0.00005
engagement_threshold = 0
scroll_threshold = 0
staleness_threshold = 10

<h2>2.10: Calculate decline_score proxy label</h2>

In [20]:
clients_last_3m["decline_score"] = (
    (clients_last_3m["ctr"] < ctr_threshold).astype(int) +
    (clients_last_3m["engagement_rate"] <= engagement_threshold).astype(int) +
    (clients_last_3m["scroll_rate"] <= scroll_threshold).astype(int) +
    (clients_last_3m["gsc_avg_position"] >= 10).astype(int) +
    (clients_last_3m["days_since_update"] > staleness_threshold).astype(int)
)

<h2>2.11: Sort decline_score proxy label in descending order for data window</h2>

In [21]:
ranked_queue = clients_last_3m.sort_values("decline_score", ascending=False)

<h2>2.12: Print top 10 ranked rows</h2>

In [22]:
print("Top 10")
ranked_queue[:10]

Top 10


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
539844,2026-05-03,client_73cda7b4e4f265ea,content_bcb7c0ea356f4316,9,12.444444,0.0,0.0,0.0,12,11-20,NaN,5
539843,2026-05-03,client_73cda7b4e4f265ea,content_f3bcaa342a4b5bc0,2,56.500000,0.0,0.0,0.0,12,21-100,NaN,5
539870,2026-05-03,client_73cda7b4e4f265ea,content_2f3e1ac0b05a336c,5,54.600000,0.0,0.0,0.0,12,21-100,NaN,5
539869,2026-05-03,client_73cda7b4e4f265ea,content_759ac8307377949d,6,33.166667,0.0,0.0,0.0,12,21-100,NaN,5
539868,2026-05-03,client_73cda7b4e4f265ea,content_bc4277fac6d48553,8,25.000000,0.0,0.0,0.0,12,21-100,NaN,5
539867,2026-05-03,client_73cda7b4e4f265ea,content_bf9fa8caa1496fd6,13,15.230769,0.0,0.0,0.0,12,11-20,NaN,5
849358,2026-05-18,client_23a62021009f63c4,content_6b3bef21b9064a31,26,32.153846,0.0,0.0,0.0,94,21-100,NaN,5
539865,2026-05-03,client_73cda7b4e4f265ea,content_3f541f5aa44f4cab,11,23.090909,0.0,0.0,0.0,12,21-100,NaN,5
479278,2026-05-04,client_73cda7b4e4f265ea,content_39fb679a0e5e64e4,20,29.150000,0.0,0.0,0.0,12,21-100,NaN,5
539862,2026-05-03,client_73cda7b4e4f265ea,content_bdb376fcb1ba45a5,1,10.000000,0.0,0.0,0.0,12,4-10,NaN,5


<h2>2.13: Save ranked queue data in 'CSV' format in 'work/outputs' directory and locak machine</h2>

In [23]:
from google.colab import files

# Ensure the output directory exists
os.makedirs('work/outputs', exist_ok=True)

# Save the ranked queue to CSV
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print(f'File saved to {output_path}. Starting download...')

# Trigger browser download to local machine
files.download(output_path)

File saved to work/outputs/baseline_action_score.csv. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

<h2>3.1: Top 20</h2>

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue[:20]

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
539844,2026-05-03,client_73cda7b4e4f265ea,content_bcb7c0ea356f4316,9,12.444444,0.0,0.0,0.0,12,11-20,NaN,5
539843,2026-05-03,client_73cda7b4e4f265ea,content_f3bcaa342a4b5bc0,2,56.500000,0.0,0.0,0.0,12,21-100,NaN,5
539870,2026-05-03,client_73cda7b4e4f265ea,content_2f3e1ac0b05a336c,5,54.600000,0.0,0.0,0.0,12,21-100,NaN,5
539869,2026-05-03,client_73cda7b4e4f265ea,content_759ac8307377949d,6,33.166667,0.0,0.0,0.0,12,21-100,NaN,5
539868,2026-05-03,client_73cda7b4e4f265ea,content_bc4277fac6d48553,8,25.000000,0.0,0.0,0.0,12,21-100,NaN,5
539867,2026-05-03,client_73cda7b4e4f265ea,content_bf9fa8caa1496fd6,13,15.230769,0.0,0.0,0.0,12,11-20,NaN,5
849358,2026-05-18,client_23a62021009f63c4,content_6b3bef21b9064a31,26,32.153846,0.0,0.0,0.0,94,21-100,NaN,5
539865,2026-05-03,client_73cda7b4e4f265ea,content_3f541f5aa44f4cab,11,23.090909,0.0,0.0,0.0,12,21-100,NaN,5
479278,2026-05-04,client_73cda7b4e4f265ea,content_39fb679a0e5e64e4,20,29.150000,0.0,0.0,0.0,12,21-100,NaN,5
539862,2026-05-03,client_73cda7b4e4f265ea,content_bdb376fcb1ba45a5,1,10.000000,0.0,0.0,0.0,12,4-10,NaN,5


<h2>3.2: Action: "Refresh_needed" because decline score is highest </h2>

<h2>3.3: Reason code: <br>
all weak signals: ctr, engagement_rate, scroll_rate, days_since_update, position > 10 for all 20 rows</h2>

<h2>3.4: Confidence note:</h2>
1. For row 1: Low confidence, only 9 impressions and days_since_update is just 12, meaning this content was very recently updated; a zero-engagement reading this soon after a change is likely too little data to judge, not a genuine decay pattern.<br>
2. For row 2: Very low confidence, only 2 impressions total; with a sample this small, zero clicks/engagement could easily be pure chance rather than a real signal.<br>
3. Row 3: Low confidence, 5 impressions is too thin a sample to confirm a genuine CTR/engagement problem; recently updated (12 days), which also weakens the "stale" part of the rule.<br>
4. Row 4: Low confidence, same pattern: small volume (6 impressions), recently updated (12 days), so the "stale" trigger is weak even though the score is maxed.<br>
5. Row 5: Low confidence, 8 impressions, recently updated; flag is likely premature given both volume and staleness are marginal<br>
6. Row 6: Low-medium confidence, slightly better volume (13 impressions) than others, but still thin, and staleness (12 days) doesn't support the "not updated for a long time" part of the rule.<br>
7. Row 7: Medium confidence, reasonable impressions (26) and a more meaningful staleness value (94 days) strengthens this case somewhat more than the 12-day rows<br>
8. Row 8: Low confidence, low volume and low staleness, same weak-evidence pattern as most rows above.<br>
9. Row 9: Low-medium confidence, moderate volume (20 impressions) but the 12-day staleness undermines the refresh rationale specifically.<br>
10. Row 10: Very low confidence, only 1 impression; this is essentially no data, the zero values here are not meaningful evidence of anything.<br>
11. Row 11: High confidence, this row has by far the strongest evidence: 90 impressions (largest volume in the top 20) and 94 days since update; a genuine zero CTR/engagement at this volume is a real pattern, not noise.<br>
12. Row 12:	Medium confidence, decent staleness (94 days) but volume (16) is still fairly thin for full confidence.
13. Row 13: Very low confidence, only 2 impressions and barely stale; weakest evidence in this set.<br>
14. Row 14: Low-medium confidence, moderate volume but low staleness weakens the case.<br>
15. Row 15: Low-medium confidence, same pattern, moderate volume, weak staleness support.<br>
16. Row 16: Low confidence, very thin volume (3 impressions) despite reasonable staleness; the zero readings could easily be sampling noise.<br>
17. ROw 17: Very low confidence, thin volume and weak staleness, one of the least trustworthy flags.<br>
18. Row 18: Medium-high confidence, solid volume (33) and meaningful staleness (94 days), a reasonably supported flag.<br>
19. Row 19: Very low confidence, very thin volume, weak staleness.<br>
20. Row 20: Low-medium confidence, decent staleness (94 days) but low volume (7 impressions) limits how much weight to put on the zero readings.


<h2>3.5: What would make it wrong?</h2>
<p>
1. Pages having 1 impression but 0 CTR and 0 Engagement rate can be by chance or maybe not but this result is not noise, its just lack of data.<br>
2. Low engagement and scroll rates can sometimes be caused by bot traffic hitting a page without interacting, which can or may skew metric.<br>
3. Certain topics naturally lose engagement at specific times of the year and this is just 60 day window of mid year months and a high decline score is expected but does not necessarily meant refresh_needed.
 </p>

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.